# Can Predictive Uncertainty Flag Unreliable Classifications of Early-Stage Pancreatic Cancer?

**BME 304 Final Project** | Rosalyn Morin & Daniel Elsharkawy

This notebook goes through the analysis step by step. Run the cells **in order, from top to bottom**.

| Step | What it does |
|---|---|
| 1 | Imports and settings |
| 2 | Load the data |
| 3 | Select eligible samples (controls, benign disease, stage I–II PDAC) |
| 4 | Explore the data and check the log transformation |
| 5 | Define the model (log-transform, standardize, logistic regression) |
| 6 | Stratified 5-fold cross-validation with 200 bootstrap models per fold |
| 7 | Overall model performance |
| 8 | RQ1: Is bootstrap variability linked to errors, and does it flag errors better than distance to the threshold? |
| 9 | RQ2: Is the effect strongest for benign disease, especially chronic pancreatitis? |
| 10 | RQ3: What does each flagging approach cost? |
| 11 | RQ4: Calibration |
| 12 | Summary |

**Data:** S1 Table from Debernardi et al., *PLOS Medicine* (2020), doi: 10.1371/journal.pmed.1003489.

## Step 1: Imports and settings

If an import fails, uncomment the `%pip` line, run it once, then restart the kernel.

In [ ]:
# %pip install pandas numpy scipy scikit-learn matplotlib openpyxl

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             balanced_accuracy_score, brier_score_loss)
from sklearn.calibration import calibration_curve

# settings used throughout the notebook
SEED = 304          # random seed so results are reproducible
N_FOLDS = 5         # outer cross-validation folds
N_BOOT = 200        # bootstrap models per fold
FLAG_PROPORTIONS = np.round(np.arange(0, 0.55, 0.05), 2)   # 0%, 5%, ..., 50% of samples flagged

# predictors (same five as PancRISK)
FEATURES = ["age", "creatinine", "LYVE1", "REG1B", "TFF1"]
LOG_FEATURES = ["creatinine", "LYVE1", "REG1B", "TFF1"]   # concentrations get log-transformed

# folders for saved figures and results
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

## Step 2: Load the data

Put the S1 Table file in a folder called `data` next to this notebook, or change `DATA_PATH` to wherever the file is. A `.csv` version works too.

In [ ]:
DATA_PATH = "data/journal.pmed.1003489.s009.xlsx"

if DATA_PATH.endswith(".csv"):
    raw = pd.read_csv(DATA_PATH)
else:
    raw = pd.read_excel(DATA_PATH)

print("Shape:", raw.shape)
print("Columns:", list(raw.columns))
raw.head()

### Match the column names

The left side is the name used in this notebook, and the right side is the column name in the file. If the check below says a column is missing, change the right-hand name to match the file.

In [ ]:
COLUMN_NAMES = {
    "id": "sample_id",
    "cohort": "patient_cohort",
    "site": "sample_origin",
    "age": "age",
    "diagnosis": "diagnosis",                  # 1 = control, 2 = benign, 3 = PDAC
    "stage": "stage",
    "benign_dx": "benign_sample_diagnosis",
    "creatinine": "creatinine",
    "LYVE1": "LYVE1",
    "REG1B": "REG1B",
    "TFF1": "TFF1",
}

missing = [col for col in COLUMN_NAMES.values() if col not in raw.columns]
if missing:
    raise ValueError(f"These columns were not found in the file: {missing}. Update COLUMN_NAMES above.")

df = raw.rename(columns={file_name: our_name for our_name, file_name in COLUMN_NAMES.items()})
print("All columns found.")

## Step 3: Select eligible samples

We keep:
- all **healthy controls** (diagnosis = 1)
- all **benign hepatobiliary disease** samples (diagnosis = 2)
- **PDAC** samples (diagnosis = 3) only if they are **stage I–II**

The outcome is `y = 1` for early-stage PDAC and `y = 0` for everyone else.

Expected counts from our proposal: **493** eligible records, **102** cancers, **16** stage I.

In [ ]:
print("Diagnosis counts (all records):")
print(df["diagnosis"].value_counts().sort_index(), "\n")

print("Stage counts among PDAC samples:")
print(df.loc[df["diagnosis"] == 3, "stage"].value_counts(dropna=False), "\n")

In [ ]:
EARLY_STAGES = ["I", "IA", "IB", "II", "IIA", "IIB"]
STAGE_I = ["I", "IA", "IB"]

stage_clean = df["stage"].astype(str).str.strip().str.upper()

is_control = df["diagnosis"] == 1
is_benign = df["diagnosis"] == 2
is_early_pdac = (df["diagnosis"] == 3) & stage_clean.isin(EARLY_STAGES)

data = df[is_control | is_benign | is_early_pdac].copy()
data["stage_clean"] = stage_clean[data.index]
data = data.reset_index(drop=True)

# outcome and group labels
data["y"] = (data["diagnosis"] == 3).astype(int)
data["group"] = data["diagnosis"].map({1: "Control", 2: "Benign", 3: "PDAC I-II"})

# flag chronic pancreatitis within the benign group
benign_text = data["benign_dx"].astype(str).str.lower()
data["chronic_pancreatitis"] = (
    (data["diagnosis"] == 2)
    & benign_text.str.contains("chronic")
    & benign_text.str.contains("pancreatitis")
)

print("Eligible records:", len(data))
print("Early-stage PDAC (y = 1):", data["y"].sum())
print("Stage I PDAC:", data["stage_clean"].isin(STAGE_I).sum())
print("Chronic pancreatitis (within benign):", data["chronic_pancreatitis"].sum())
print()
print(data["group"].value_counts())

### Check for missing values in the predictors

Our proposal found the five predictors complete for all 493 eligible records, so every count below should be 0.

In [ ]:
print(data[FEATURES].isna().sum())
print("\nAny values <= 0 in the concentrations?")
print((data[LOG_FEATURES] <= 0).sum())

## Step 4: Explore the data

Biomarker concentrations are usually right-skewed. The log transformation should make them more symmetric, which suits logistic regression better.

We use `log(1 + x)` so that any values of 0 don't cause an error.

In [ ]:
fig, axes = plt.subplots(2, len(LOG_FEATURES), figsize=(16, 7))

for i, col in enumerate(LOG_FEATURES):
    for grp, color in zip(["Control", "Benign", "PDAC I-II"], ["tab:blue", "tab:orange", "tab:red"]):
        values = data.loc[data["group"] == grp, col]
        axes[0, i].hist(values, bins=30, alpha=0.5, color=color, label=grp)
        axes[1, i].hist(np.log1p(values), bins=30, alpha=0.5, color=color, label=grp)
    axes[0, i].set_title(f"{col} (raw)")
    axes[1, i].set_title(f"log(1 + {col})")

axes[0, 0].legend()
plt.tight_layout()
plt.savefig("figures/biomarker_distributions.png", dpi=200)
plt.show()

In [ ]:
# median of each predictor by group
data.groupby("group")[FEATURES].median().round(3)

## Step 5: Define the model

Each model is a **pipeline** with three steps:
1. Log-transform the four concentrations (age is left as is).
2. Standardize all five predictors.
3. Fit a logistic regression with no regularization, like the published PancRISK model. (A very large `C` turns regularization off; this works on old and new versions of scikit-learn.)

Because the scaler is inside the pipeline, it is fit **only on the training data** every time the model is fit. This prevents data leakage.

In [ ]:
def make_model():
    preprocess = ColumnTransformer([
        ("log", FunctionTransformer(np.log1p), LOG_FEATURES),
        ("keep", "passthrough", ["age"]),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("scale", StandardScaler()),
        ("logreg", LogisticRegression(C=1e10, max_iter=5000)),   # very large C = effectively no regularization
    ])

### Choosing the classification threshold

The threshold is chosen **using only the training fold**:
1. Run an inner 5-fold cross-validation on the training fold to get out-of-sample probabilities.
2. Pick the threshold that maximizes **Youden's J** (sensitivity + specificity − 1).

This keeps the test fold completely unseen.

In [ ]:
def choose_threshold(X_train, y_train, seed):
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    p_inner = cross_val_predict(make_model(), X_train, y_train,
                                cv=inner_cv, method="predict_proba")[:, 1]
    fpr, tpr, thresholds = roc_curve(y_train, p_inner)
    youden_j = tpr - fpr
    return thresholds[np.argmax(youden_j)]

## Step 6: Cross-validation with bootstrap uncertainty

For each of the 5 outer folds:
1. Fit the main model on the training fold and predict the held-out test fold.
2. Choose the threshold from the training fold (Step 5).
3. Fit **200 bootstrap models**, each on a resample (with replacement) of the training fold, and predict the test fold with each.
4. For each test sample, record:
   - `p`: the main model's predicted probability of cancer
   - `boot_sd`: the **standard deviation** of the 200 bootstrap predictions (our uncertainty measure)
   - `dist_to_threshold`: how far `p` is from the threshold (the simple baseline)

The folds are stratified by **diagnosis group** (control / benign / PDAC), so every fold has the same mix of all three.

This takes about a minute or two.

In [ ]:
X = data[FEATURES]
y = data["y"].values

outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
rng = np.random.default_rng(SEED)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, data["group"]), start=1):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_test = X.iloc[test_idx]

    # 1. main model
    model = make_model().fit(X_train, y_train)
    p_test = model.predict_proba(X_test)[:, 1]

    # 2. threshold from the training fold only
    threshold = choose_threshold(X_train, y_train, seed=SEED + fold)

    # 3. bootstrap models
    boot_preds = np.zeros((N_BOOT, len(test_idx)))
    for b in range(N_BOOT):
        resample = rng.integers(0, len(train_idx), size=len(train_idx))
        boot_model = make_model().fit(X_train.iloc[resample], y_train[resample])
        boot_preds[b] = boot_model.predict_proba(X_test)[:, 1]

    # 4. store results for each test sample
    fold_df = data.iloc[test_idx][["id", "group", "chronic_pancreatitis", "y"]].copy()
    fold_df["fold"] = fold
    fold_df["p"] = p_test
    fold_df["threshold"] = threshold
    fold_df["pred"] = (p_test >= threshold).astype(int)
    fold_df["boot_mean"] = boot_preds.mean(axis=0)
    fold_df["boot_sd"] = boot_preds.std(axis=0, ddof=1)
    fold_df["dist_to_threshold"] = np.abs(p_test - threshold)
    fold_results.append(fold_df)

    print(f"Fold {fold}: {len(train_idx)} train, {len(test_idx)} test, "
          f"{y[test_idx].sum()} cancers in test, threshold = {threshold:.3f}")

results = pd.concat(fold_results).sort_index()
results["error"] = (results["pred"] != results["y"]).astype(int)

results.to_csv("results/cv_predictions.csv", index=False)
results.head()

## Step 7: Overall model performance

Metrics are calculated on each held-out fold, then summarized as **mean ± SD across the 5 folds**.

In [ ]:
def fold_metrics(d):
    tn, fp, fn, tp = confusion_matrix(d["y"], d["pred"], labels=[0, 1]).ravel()
    return {
        "AUROC": roc_auc_score(d["y"], d["p"]),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Balanced accuracy": balanced_accuracy_score(d["y"], d["pred"]),
        "Threshold": d["threshold"].iloc[0],
    }

per_fold = pd.DataFrame([fold_metrics(results[results["fold"] == f]) for f in range(1, N_FOLDS + 1)],
                        index=pd.Index(range(1, N_FOLDS + 1), name="Fold"))

summary = pd.DataFrame({"Mean": per_fold.mean(), "SD": per_fold.std()})
print(per_fold.round(3), "\n")
print(summary.round(3))
per_fold.to_csv("results/per_fold_metrics.csv")

In [ ]:
# confusion matrix over all held-out predictions
cm = confusion_matrix(results["y"], results["pred"], labels=[0, 1])
print("Confusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(cm, index=["True: no cancer", "True: cancer"],
                   columns=["Pred: no cancer", "Pred: cancer"]))
print("\nTotal errors:", results["error"].sum(), "of", len(results),
      f"({results['error'].mean():.1%})")

### AUROC by comparison (vs. the original paper)

The paper reported AUCs of **0.936** (healthy controls vs. early PDAC), **0.871** (benign vs. early PDAC) and **0.818** (chronic pancreatitis vs. early PDAC). These use all held-out predictions pooled together.

In [ ]:
pdac = results["y"] == 1
comparisons = {
    "Control vs PDAC I-II": results["group"] == "Control",
    "Benign vs PDAC I-II": results["group"] == "Benign",
    "Chronic pancreatitis vs PDAC I-II": results["chronic_pancreatitis"],
}

rows = []
for name, comparison_group in comparisons.items():
    subset = results[comparison_group | pdac]
    rows.append({"Comparison": name,
                 "n non-cancer": int(comparison_group.sum()),
                 "AUROC": roc_auc_score(subset["y"], subset["p"])})

auc_table = pd.DataFrame(rows)
auc_table["AUROC"] = auc_table["AUROC"].round(3)
auc_table

## Step 8: RQ1 — Is bootstrap variability linked to errors?

**RQ1.** Are incorrect classifications associated with higher bootstrap variability, and when both approaches set aside the same proportion of samples, does flagging by bootstrap variability produce a lower error rate among the remaining predictions than flagging by distance from the decision threshold?

### 8a. Do wrong predictions have higher bootstrap variability?

- **Mann–Whitney U test** (one-sided): is `boot_sd` higher for wrong predictions than correct ones? We use a non-parametric test because uncertainty values are usually skewed.
- **AUROC for error detection**: how well each measure ranks wrong predictions above correct ones (0.5 = no better than chance). For distance to the threshold, a *smaller* distance means *less* sure, so we use the negative distance.

In [ ]:
wrong = results[results["error"] == 1]
right = results[results["error"] == 0]

u_stat, p_value = stats.mannwhitneyu(wrong["boot_sd"], right["boot_sd"], alternative="greater")

print(f"Median bootstrap SD, wrong predictions:   {wrong['boot_sd'].median():.4f}  (n = {len(wrong)})")
print(f"Median bootstrap SD, correct predictions: {right['boot_sd'].median():.4f}  (n = {len(right)})")
print(f"Mann-Whitney U = {u_stat:.1f}, one-sided p = {p_value:.4g}\n")

auc_boot = roc_auc_score(results["error"], results["boot_sd"])
auc_dist = roc_auc_score(results["error"], -results["dist_to_threshold"])
print(f"AUROC for detecting errors, bootstrap SD:          {auc_boot:.3f}")
print(f"AUROC for detecting errors, distance to threshold: {auc_dist:.3f}")

rho, rho_p = stats.spearmanr(results["boot_sd"], results["dist_to_threshold"])
print(f"\nSpearman correlation between bootstrap SD and distance to threshold: {rho:.3f} (p = {rho_p:.3g})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].boxplot([right["boot_sd"], wrong["boot_sd"]])
axes[0].set_xticks([1, 2], ["Correct", "Wrong"])
axes[0].set_ylabel("Bootstrap SD of predicted probability")
axes[0].set_title("Bootstrap variability by prediction outcome")

colors = np.where(results["error"] == 1, "tab:red", "tab:gray")
axes[1].scatter(results["dist_to_threshold"], results["boot_sd"], c=colors, s=15, alpha=0.7)
axes[1].set_xlabel("Distance from threshold")
axes[1].set_ylabel("Bootstrap SD")
axes[1].set_title("Two uncertainty measures (red = wrong prediction)")

plt.tight_layout()
plt.savefig("figures/rq1_uncertainty_vs_errors.png", dpi=200)
plt.show()

### 8b. Flagging at the same proportion

For each flagging rate (0% to 50%), each method sets aside the same number of samples:
- **Bootstrap variability:** flag the samples with the **highest** `boot_sd`.
- **Distance to threshold:** flag the samples **closest** to the threshold.

Then we compute the **error rate among the unflagged predictions**. Lower is better. The dashed line is the overall error rate with no flagging.

In [ ]:
def flag_top(scores, proportion):
    # flag the given proportion of samples with the highest scores
    n_flag = int(round(proportion * len(scores)))
    order = np.argsort(-scores, kind="stable")
    flagged = np.zeros(len(scores), dtype=bool)
    flagged[order[:n_flag]] = True
    return flagged

methods = {
    "Bootstrap variability": results["boot_sd"].values,
    "Distance to threshold": -results["dist_to_threshold"].values,   # closest = highest score
}

y_true = results["y"].values
y_pred = results["pred"].values
is_error = results["error"].values.astype(bool)

flag_rows = []
for proportion in FLAG_PROPORTIONS:
    for method, scores in methods.items():
        flagged = flag_top(scores, proportion)
        kept = ~flagged
        flag_rows.append({
            "Method": method,
            "Proportion flagged": proportion,
            "Number flagged": int(flagged.sum()),
            "Error rate (unflagged)": is_error[kept].mean(),
            "Errors caught": int((flagged & is_error).sum()),
            "Cancers flagged": int((flagged & (y_true == 1)).sum()),
            "False negatives unflagged": int((kept & (y_true == 1) & (y_pred == 0)).sum()),
        })

flagging = pd.DataFrame(flag_rows)
flagging.to_csv("results/flagging_comparison.csv", index=False)

pivot = flagging.pivot(index="Proportion flagged", columns="Method", values="Error rate (unflagged)")
pivot.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
for method, marker in zip(methods, ["o", "s"]):
    d = flagging[flagging["Method"] == method]
    plt.plot(d["Proportion flagged"] * 100, d["Error rate (unflagged)"] * 100, marker=marker, label=method)

plt.axhline(results["error"].mean() * 100, color="gray", linestyle="--", label="No flagging")
plt.xlabel("Samples flagged for review (%)")
plt.ylabel("Error rate among unflagged predictions (%)")
plt.title("RQ1: Error rate after flagging")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("figures/rq1_flagging_curves.png", dpi=200)
plt.show()

## Step 9: RQ2 — Is the effect strongest for benign disease?

**RQ2.** Is this effect strongest when distinguishing cancer from benign hepatobiliary disease, particularly chronic pancreatitis?

We compare error rates and bootstrap variability across groups:
- **Kruskal–Wallis test:** does bootstrap SD differ across control, benign and PDAC?
- **Mann–Whitney U test:** within benign disease, is bootstrap SD higher for chronic pancreatitis than for other benign conditions?

The chronic pancreatitis subgroup is small, so interpret its results with caution.

In [ ]:
def subgroup(row):
    if row["group"] == "Benign":
        return "Chronic pancreatitis" if row["chronic_pancreatitis"] else "Other benign"
    return row["group"]

results["subgroup"] = results.apply(subgroup, axis=1)
order = ["Control", "Other benign", "Chronic pancreatitis", "PDAC I-II"]

group_table = (results.groupby("subgroup")
               .agg(n=("y", "size"),
                    error_rate=("error", "mean"),
                    median_boot_sd=("boot_sd", "median"),
                    mean_boot_sd=("boot_sd", "mean"))
               .reindex(order))
group_table.round(4)

In [ ]:
groups = [results.loc[results["group"] == g, "boot_sd"] for g in ["Control", "Benign", "PDAC I-II"]]
h_stat, kw_p = stats.kruskal(*groups)
print(f"Kruskal-Wallis across Control / Benign / PDAC I-II: H = {h_stat:.2f}, p = {kw_p:.4g}")

cp = results.loc[results["subgroup"] == "Chronic pancreatitis", "boot_sd"]
other = results.loc[results["subgroup"] == "Other benign", "boot_sd"]
if len(cp) > 0 and len(other) > 0:
    u_cp, p_cp = stats.mannwhitneyu(cp, other, alternative="greater")
    print(f"Chronic pancreatitis vs other benign (one-sided): U = {u_cp:.1f}, p = {p_cp:.4g}  "
          f"(n = {len(cp)} vs {len(other)})")
else:
    print("No chronic pancreatitis samples found. Check the benign_dx column in Step 3.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([results.loc[results["subgroup"] == g, "boot_sd"] for g in order])
axes[0].set_xticks(range(1, len(order) + 1), order)
axes[0].set_ylabel("Bootstrap SD")
axes[0].set_title("Bootstrap variability by group")

axes[1].bar(order, group_table["error_rate"] * 100, color=["tab:blue", "tab:orange", "tab:brown", "tab:red"])
axes[1].set_ylabel("Error rate (%)")
axes[1].set_title("Error rate by group")

plt.tight_layout()
plt.savefig("figures/rq2_groups.png", dpi=200)
plt.show()

## Step 10: RQ3 — What does each flagging approach cost?

**RQ3.** What does each flagging approach cost in practice, measured by the number of cancer cases sent for further review and the number of false negatives that remain unflagged?

A lower error rate is only useful if we can see what it costs: flagged cancers need extra follow-up, and any **false negatives left unflagged** are missed cancers.

In [ ]:
total_fn = int(((y_true == 1) & (y_pred == 0)).sum())
print(f"Total cancers: {int(y_true.sum())},  false negatives with no flagging: {total_fn}\n")

cost_table = flagging[flagging["Proportion flagged"].isin([0.1, 0.2, 0.3])][
    ["Proportion flagged", "Method", "Number flagged", "Errors caught",
     "Cancers flagged", "False negatives unflagged", "Error rate (unflagged)"]
]
cost_table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for method, marker in zip(methods, ["o", "s"]):
    d = flagging[flagging["Method"] == method]
    axes[0].plot(d["Proportion flagged"] * 100, d["Cancers flagged"], marker=marker, label=method)
    axes[1].plot(d["Proportion flagged"] * 100, d["False negatives unflagged"], marker=marker, label=method)

axes[0].set_title("Cancer cases sent for review")
axes[0].set_ylabel("Number of cancers flagged")
axes[1].set_title("Missed cancers left unflagged")
axes[1].set_ylabel("False negatives unflagged")
for ax in axes:
    ax.set_xlabel("Samples flagged for review (%)")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("figures/rq3_costs.png", dpi=200)
plt.show()

## Step 11: RQ4 — Calibration

**RQ4.** How well do the model's predicted probabilities agree with observed cancer frequencies within this dataset?

- **Calibration curve:** groups samples by predicted probability and compares the average prediction with the actual fraction of cancers in each group. A perfectly calibrated model follows the diagonal.
- **Brier score:** the mean squared difference between predicted probability and the true outcome (lower is better).

This dataset is a case–control study, so the share of cancers (about 21%) is much higher than in the general population. Calibration here only applies within this dataset.

In [ ]:
frac_cancer, mean_pred = calibration_curve(results["y"], results["p"], n_bins=8, strategy="quantile")
brier = brier_score_loss(results["y"], results["p"])
print(f"Brier score: {brier:.4f}")
print(f"For reference, always predicting the cancer rate ({results['y'].mean():.3f}) "
      f"gives a Brier score of {brier_score_loss(results['y'], np.full(len(results), results['y'].mean())):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot([0, 1], [0, 1], "k--", label="Perfect calibration")
axes[0].plot(mean_pred, frac_cancer, "o-", label="Logistic regression")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed fraction of cancers")
axes[0].set_title(f"Calibration curve (Brier = {brier:.3f})")
axes[0].legend()

axes[1].hist(results.loc[results["y"] == 0, "p"], bins=25, alpha=0.6, label="No cancer")
axes[1].hist(results.loc[results["y"] == 1, "p"], bins=25, alpha=0.6, label="PDAC I-II")
axes[1].set_xlabel("Predicted probability of cancer")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of predicted probabilities")
axes[1].legend()

plt.tight_layout()
plt.savefig("figures/rq4_calibration.png", dpi=200)
plt.show()

## Step 12: Summary of key results

In [ ]:
print("MODEL PERFORMANCE (mean ± SD across folds)")
for metric in ["AUROC", "Sensitivity", "Specificity", "Balanced accuracy"]:
    print(f"  {metric}: {per_fold[metric].mean():.3f} ± {per_fold[metric].std():.3f}")

print("\nRQ1: UNCERTAINTY AND ERRORS")
print(f"  Mann-Whitney (wrong > correct bootstrap SD): p = {p_value:.4g}")
print(f"  Error-detection AUROC: bootstrap SD = {auc_boot:.3f}, distance to threshold = {auc_dist:.3f}")
for proportion in [0.1, 0.2]:
    row = pivot.loc[proportion]
    print(f"  Error rate after flagging {proportion:.0%}: "
          f"bootstrap = {row['Bootstrap variability']:.3f}, distance = {row['Distance to threshold']:.3f}")

print("\nRQ2: GROUPS")
print(f"  Kruskal-Wallis p = {kw_p:.4g}")
print(group_table[["n", "error_rate", "median_boot_sd"]].round(4).to_string())

print(f"\nRQ4: CALIBRATION\n  Brier score = {brier:.4f}")